# 02: Missed pairs and address comparison
Step 1: why does name-token blocking miss true matches? Step 2: which address parts agree on true pairs?

In [1]:
import os, re, unicodedata
os.environ['TMP']=os.environ['TEMP']='D:/tmp'
import pandas as pd, numpy as np
from collections import defaultdict, Counter
from rapidfuzz import fuzz
pd.set_option('display.max_colwidth',70); pd.set_option('display.width',250)
W='D:/Amazon_ML_Challenge/work/'
s1=pd.read_parquet(W+'dev_s1.parquet'); s2=pd.read_parquet(W+'dev_s2.parquet'); s3=pd.read_parquet(W+'dev_s3.parquet'); gt=pd.read_parquet(W+'dev_gt.parquet')
print(list(s1.columns))
SUFFIX=r"\b(private|pvt|limited|ltd|llp|l l p|llc|l l c|inc|incorporated|corp|corporation|co|company|the|and)\b"
def norm_name(x):
    x=unicodedata.normalize('NFKD',str(x)).lower()
    x=re.sub(r"^(india|us|usa)\s{2,}","",x)
    x=re.sub(r"\.(com|in|net|org|co\.in)\b","",x)
    x=re.sub(r"[^a-z0-9 ]"," ",x); x=re.sub(SUFFIX," ",x)
    return re.sub(r"\s+"," ",x).strip()
for d in (s1,s2,s3): d['nn']=d.business_name.map(norm_name)
CAP=200
def build_index(d):
    idx=defaultdict(list)
    for i,(n,c) in enumerate(zip(d.nn,d.country)):
        for t in set(n.split()): idx[(c,t)].append(i)
    return idx
idx={'S2':build_index(s2),'S3':build_index(s3)}
D={'S2':s2.reset_index(drop=True),'S3':s3.reset_index(drop=True)}
pos={k:dict(zip(D[k].entity_id,range(len(D[k])))) for k in D}
q=s1.sample(20000,random_state=0).reset_index(drop=True)
g=gt.set_index('source1_entity_id').matched_entity_ids.str.split(',')

['entity_id', 'business_name', 'business_address', 'country']


## Step 1: categorize missed true pairs

In [2]:
rows=[]
for _,r in q.iterrows():
    toks=set(r.nn.split())
    for src in ('S2','S3'):
        d=D[src]; ix=idx[src]
        cand=set()
        for t in toks: 
            l=ix.get((r.country,t),())
            if len(l)<=CAP: cand.update(l)
        for tid in g[r.entity_id]:
            if not tid.startswith(src) or tid not in pos[src]: continue
            j=pos[src][tid]
            if j in cand: continue
            m=d.iloc[j]; mt=set(m.nn.split()); shared=toks&mt
            sim=fuzz.token_sort_ratio(r.nn,m.nn)
            if not toks or not mt: cat='empty_name_after_norm'
            elif shared and all(len(ix.get((r.country,t),()))>CAP for t in shared): cat='only_common_tokens'
            elif shared: cat='shared_token_but_missed?'
            elif sim>=80: cat='typo_or_spacing'
            elif re.search(r"\.\w{2,3}\b",m.business_name+r.business_name): cat='domain_style'
            elif sim>=50: cat='partly_similar'
            else: cat='totally_different_name'
            rows.append((src,cat,r.business_name,m.business_name,r.business_address,m.business_address,r.country,sim))
miss=pd.DataFrame(rows,columns=['src','cat','n1','n2','a1','a2','country','sim'])
print(len(miss),'missed pairs'); print(miss.groupby(['src','cat']).size().unstack(0).fillna(0).astype(int))
print(miss.groupby(['country','cat']).size().unstack(0).fillna(0).astype(int))

33078 missed pairs


src                        S2     S3
cat                                 
domain_style               21     32
empty_name_after_norm    3153   1717
only_common_tokens      11901  10739
partly_similar           1192   1166
totally_different_name    782   1089
typo_or_spacing           601    685
country                 India     US
cat                                 
domain_style               14     39
empty_name_after_norm    4870      0
only_common_tokens       6763  15877
partly_similar            813   1545
totally_different_name    769   1102
typo_or_spacing           560    726


In [3]:
for cat,x in miss.groupby('cat'):
    print('\n=====',cat,len(x)); print(x.sample(min(5,len(x)),random_state=0)[['n1','n2','a1','a2']].to_string(index=False))


===== domain_style 53
                          n1                                  n2                                                                                a1                                                                                          a2
       Golden Materials P.C.     goldenmaterials.com (ID: 88096)                                                 358 Conley Circle, Montgomery, AL                                                           358 CONLEY CIRCLE, MONTGOMERY, AL
              # 7 HB Banking                             7hb.com                                               124 Grassy Plain Street, Bethel, CT                                       Connecticut, 124 Grassy Plain St, Bethel, PO Box 3262
 Gregerson Transalta Company gregersontransalta.com - 5513043619                                                 6311 Sherlock Way, KY, Louisville                                                     6311 Sherlock Way, Louisville, Kentucky
            Welcome P

## Step 2: address agreement on TRUE pairs

In [4]:
def postal(a,c):
    a=str(a)
    m=re.findall(r"\b\d{6}\b",a) if c=='India' else re.findall(r"\b\d{5}\b",a)
    return m[-1] if m else None
def atoks(a): return set(re.sub(r"[^a-z0-9 ]"," ",unicodedata.normalize('NFKD',str(a)).lower()).split())
rec=[]
for _,r in q.sample(6000,random_state=1).iterrows():
    for src in ('S2','S3'):
        for tid in g[r.entity_id]:
            if not tid.startswith(src) or tid not in pos[src]: continue
            m=D[src].iloc[pos[src][tid]]
            p1,p2=postal(r.business_address,r.country),postal(m.business_address,r.country)
            t1,t2=atoks(r.business_address),atoks(m.business_address)
            rec.append((src,r.country,p1 is not None,p2 is not None,p1 is not None and p1==p2,len(t1&t2)/max(1,len(t1|t2)),bool(t1&t2),r.business_address,m.business_address))
A=pd.DataFrame(rec,columns=['src','country','has_p1','has_p2','p_match','jacc','any_tok','a1','a2'])
print(A.groupby(['country','src'])[['has_p1','has_p2','p_match','any_tok']].mean().round(3))
print(A.groupby(['country','src']).jacc.describe().round(3))

             has_p1  has_p2  p_match  any_tok
country src                                  
India   S2    0.000   0.000    0.000    0.959
        S3    0.000   0.000    0.000    0.960
US      S2    0.117   0.103    0.080    0.950
        S3    0.120   0.105    0.083    0.952
              count   mean    std  min    25%    50%    75%  max
country src                                                     
India   S2   4102.0  0.773  0.243  0.0  0.700  0.846  0.933  1.0
        S3   4364.0  0.620  0.269  0.0  0.400  0.692  0.824  1.0
US      S2   5901.0  0.631  0.253  0.0  0.500  0.667  0.750  1.0
        S3   6229.0  0.480  0.224  0.0  0.333  0.444  0.625  1.0


In [5]:
for (c,s),x in A.groupby(['country','src']):
    print('\n=====',c,s); print(x.sample(6,random_state=0)[['a1','a2']].to_string(index=False))


===== India S2
                                                                                                                            a1                                                                                                                     a2
                                                    19-7/8 Nilaakkal Veedu, Nariyanvilai, Vilavancode, Kanyakumari, Tamil Nadu                                                                        19-7/8 NILAAKKAL VEEDU, KANYAKUMARI, Tamil Nadu
                                       Suryanarayan Dwivedi Kaushambi, Surseni Tilhapur, Newada Hail, Kaushambi, Uttar Pradesh                             82 SURYANARAYAN DWIVEDI KAUSHAMBI, SURSENI TILHAPUR, NEWADA HAIL, KAUSHAMBI, Uttar Pradesh
1107-A, The Platina, Tanvi Complex, Next To S V Road, Near Dahisar Petrol Pump, Dahisar (E), Dahisar East, Mumbai, Maharashtra 1107-A, THE PLATINA, TANVI COMPLEX, NEXT TO S V ROAD, NEAR DAHISAR PETROL PUMP, DAHISAR (E), DAHISAR EAST, Maharashtra


## Step 2b: does address alone rescue the missed pairs?
For missed pairs, does the pair share the postal code or a city-like token?

In [6]:
miss['p_match']=[ (postal(a,c) is not None and postal(a,c)==postal(b,c)) for a,b,c in zip(miss.a1,miss.a2,miss.country)]
miss['any_tok']=[bool(atoks(a)&atoks(b)) for a,b in zip(miss.a1,miss.a2)]
print(miss.groupby('cat')[['p_match','any_tok']].mean().round(3))
print('overall share of missed pairs with same postal code:',miss.p_match.mean().round(3))

                        p_match  any_tok
cat                                     
domain_style              0.019    1.000
empty_name_after_norm     0.000    1.000
only_common_tokens        0.051    0.950
partly_similar            0.047    1.000
totally_different_name    0.048    1.000
typo_or_spacing           0.050    0.999
overall share of missed pairs with same postal code: 0.043
